In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nyc_sales_property_cleaned.csv")

print("Shape:", df.shape)
df.head()

Shape: (650335, 20)


,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AS OF FINAL ROLL,BLOCK,LOT,BUILDING CLASS AS OF FINAL ROLL,ADDRESS,APARTMENT NUMBER,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,1,CHINATOWN,47 CONDO NON-BUSINESS STORAGE,4,172,1490,RS,"50 FRANKLIN STREET, S9B",S9B,10013.0,0.0,1.0,1.0,0.0,0.0,2007.0,4,RS,1590000,2019-06-21
1,1,FASHION,21 OFFICE BUILDINGS,4,779,23,O6,227 WEST 29TH STREET,NaN,10001.0,0.0,3.0,3.0,NaN,NaN,1927.0,4,O6,0,2019-05-03
2,1,FASHION,13 CONDOS - ELEVATOR APARTMENTS,2,841,1044,R4,"16 WEST 40TH STREET, 26A",26A,10018.0,1.0,0.0,1.0,0.0,0.0,2015.0,2,R4,5203258,2019-03-11
3,1,CHELSEA,22 STORE BUILDINGS,4,774,79,K4,276 WEST 25TH STREET,NaN,10001.0,0.0,2.0,2.0,NaN,NaN,1910.0,4,K4,6900000,2019-08-02
4,1,CHELSEA,21 OFFICE BUILDINGS,4,802,75,O6,158 WEST 27 STREET,NaN,10001.0,0.0,14.0,14.0,NaN,NaN,1913.0,4,O6,99350000,2019-10-24


In [22]:
df["SALE YEAR"] = pd.to_datetime(df["SALE DATE"]).dt.year

df["PROPERTY AGE"] = df["SALE YEAR"] - df["YEAR BUILT"]

df["PRICE PER SQ FT"] = np.where(
    (df["GROSS SQUARE FEET"] > 0) & (df["SALE PRICE"] > 0),
    df["SALE PRICE"] / df["GROSS SQUARE FEET"],
    np.nan
)

print([
    "SALE YEAR",
    "PROPERTY AGE",
    "PRICE PER SQ FT"
])

df[["SALE DATE", "SALE YEAR", "YEAR BUILT", "PROPERTY AGE", "SALE PRICE", "GROSS SQUARE FEET", "PRICE PER SQ FT"]].head()

['SALE YEAR', 'PROPERTY AGE', 'PRICE PER SQ FT']


,SALE DATE,SALE YEAR,YEAR BUILT,PROPERTY AGE,SALE PRICE,GROSS SQUARE FEET,PRICE PER SQ FT
0,2019-06-21,2019,2007.0,12.0,1590000,0.0,NaN
1,2019-05-03,2019,1927.0,92.0,0,NaN,NaN
2,2019-03-11,2019,2015.0,4.0,5203258,0.0,NaN
3,2019-08-02,2019,1910.0,109.0,6900000,NaN,NaN
4,2019-10-24,2019,1913.0,106.0,99350000,NaN,NaN


In [23]:
print("Sales years:")
print(df["SALE YEAR"].value_counts().sort_index())

print("\nBoroughs:")
print(df["BOROUGH"].value_counts().sort_index())

print("\nProperty age:")
print(df["PROPERTY AGE"].describe())

print("\nSale Price:")
print(df["SALE PRICE"].describe())

Sales years:
SALE YEAR
2016    80068
2017    79674
2018    76659
2019    61222
2020    51061
2021    69256
2022    65018
2023    53318
2024    54812
2025    59247
Name: count, dtype: int64

Boroughs:
BOROUGH
1    104926
2     55386
3    196231
4    215624
5     78168
Name: count, dtype: int64

Property age:
count    650335.000000
mean         68.446165
std          36.532838
min           0.000000
25%          39.000000
50%          77.000000
75%          97.000000
max         337.000000
Name: PROPERTY AGE, dtype: float64

Sale Price:
count    6.503350e+05
mean     1.275999e+06
std      1.110867e+07
min      0.000000e+00
25%      0.000000e+00
50%      4.929000e+05
75%      9.450000e+05
max      2.397502e+09
Name: SALE PRICE, dtype: float64


In [24]:
print("Extreme property age records:")
print(
    df.loc[
        df["PROPERTY AGE"] > 200,
        [
            "BOROUGH",
            "NEIGHBORHOOD",
            "BUILDING CLASS CATEGORY",
            "YEAR BUILT",
            "SALE YEAR",
            "PROPERTY AGE",
            "SALE PRICE"
        ]
    ].sort_values("PROPERTY AGE", ascending=False).head(20)
)

print("\nExtreme sale price records:")
print(
    df.loc[
        df["SALE PRICE"] > 100_000_000,
        [
            "BOROUGH",
            "NEIGHBORHOOD",
            "BUILDING CLASS CATEGORY",
            "YEAR BUILT",
            "SALE YEAR",
            "PROPERTY AGE",
            "SALE PRICE"
        ]
    ].sort_values("SALE PRICE", ascending=False).head(20)
)

Extreme property age records:
        BOROUGH        NEIGHBORHOOD            BUILDING CLASS CATEGORY  \
290846        5         PRINCES BAY            01 ONE FAMILY DWELLINGS   
610196        3          OCEAN HILL             14 RENTALS - 4-10 UNIT   
604021        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
615090        1        CIVIC CENTER  15 CONDOS - 2-10 UNIT RESIDENTIAL   
615091        1        CIVIC CENTER  15 CONDOS - 2-10 UNIT RESIDENTIAL   
604017        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
619733        1                SOHO             14 RENTALS - 4-10 UNIT   
604024        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604023        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604025        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604022        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604020        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604019  

In [25]:
sales_by_year=(
    df.groupby("SALE YEAR").size().reset_index(name="TRANSACTION COUNT")
)

print(sales_by_year)

   SALE YEAR  TRANSACTION COUNT
0       2016              80068
1       2017              79674
2       2018              76659
3       2019              61222
4       2020              51061
5       2021              69256
6       2022              65018
7       2023              53318
8       2024              54812
9       2025              59247


In [26]:
sales_by_borough = (
    df.groupby("BOROUGH").size().reset_index(name="TRANSACTION COUNT").sort_values("TRANSACTION COUNT", ascending=False)
)

print(sales_by_borough)

   BOROUGH  TRANSACTION COUNT
3        4             215624
2        3             196231
0        1             104926
4        5              78168
1        2              55386


In [27]:
borough_names = {
    1: "Manhattan",
    2: "Bronx",
    3: "Brooklyn",
    4: "Queens",
    5: "Staten Island"
}

df["BOROUGH NAME"] = df["BOROUGH"].map(borough_names)

print(df["BOROUGH NAME"].value_counts())

BOROUGH NAME
Queens           215624
Brooklyn         196231
Manhattan        104926
Staten Island     78168
Bronx             55386
Name: count, dtype: int64


In [28]:
sales_by_neighborhood = (
    df.groupby("NEIGHBORHOOD").size().reset_index(name="TRANSACTION COUNT").sort_values("TRANSACTION COUNT", ascending=False)
)

print(sales_by_neighborhood.head(20))

                NEIGHBORHOOD  TRANSACTION COUNT
91            FLUSHING-NORTH              22500
15        BEDFORD STUYVESANT              12348
24              BOROUGH PARK              11808
8                    ASTORIA               9920
13                   BAYSIDE               9499
235  UPPER EAST SIDE (59-79)               8737
72             EAST NEW YORK               8236
236  UPPER EAST SIDE (79-96)               7993
84             FLATBUSH-EAST               7948
198            RICHMOND HILL               7753
106              GREAT KILLS               7655
142         LONG ISLAND CITY               7339
39                  CANARSIE               7338
238  UPPER WEST SIDE (59-79)               7219
61             CROWN HEIGHTS               6906
11                 BAY RIDGE               6677
157             MIDTOWN WEST               6669
76                  ELMHURST               6654
92            FLUSHING-SOUTH               6604
213           SHEEPSHEAD BAY            

In [29]:
sales_by_property_type = (
    df.groupby("BUILDING CLASS CATEGORY").size().reset_index(name="TRANSACTION COUNT").sort_values("TRANSACTION COUNT", ascending=False)
)

print(sales_by_property_type.head(20))

              BUILDING CLASS CATEGORY  TRANSACTION COUNT
0             01 ONE FAMILY DWELLINGS             184881
1             02 TWO FAMILY DWELLINGS             157242
13    13 CONDOS - ELEVATOR APARTMENTS             101952
2           03 THREE FAMILY DWELLINGS              44191
9      10 COOPS - ELEVATOR APARTMENTS              40184
6      07 RENTALS - WALKUP APARTMENTS              29885
3               04 TAX CLASS 1 CONDOS              12490
15  15 CONDOS - 2-10 UNIT RESIDENTIAL              12071
19                 22 STORE BUILDINGS               8818
8        09 COOPS - WALKUP APARTMENTS               8457
12      12 CONDOS - WALKUP APARTMENTS               6707
14             14 RENTALS - 4-10 UNIT               6138
40                   44 CONDO PARKING               4899
7    08 RENTALS - ELEVATOR APARTMENTS               4447
18                21 OFFICE BUILDINGS               3444
17                     17 CONDO COOPS               3098
26                      30 WARE

In [31]:
property_type_value = (
    df[df["SALE PRICE"] > 0].groupby("BUILDING CLASS CATEGORY")["SALE PRICE"]
    .median().reset_index(name="MEDIAN SALE PRICE").sort_values("MEDIAN SALE PRICE", ascending=False)
)

print(property_type_value.head(20))

                      BUILDING CLASS CATEGORY  MEDIAN SALE PRICE
21                           25 LUXURY HOTELS         55015506.0
30                                34 THEATRES         16625000.0
20                          23 LOFT BUILDINGS         10500000.0
7            08 RENTALS - ELEVATOR APARTMENTS          7735000.0
28          32 HOSPITAL AND HEALTH FACILITIES          5803750.0
27                  31 COMMERCIAL VACANT LAND          4000000.0
29                  33 EDUCATIONAL FACILITIES          3925000.0
23                               27 FACTORIES          3400000.0
26                              30 WAREHOUSES          3350000.0
18                        21 OFFICE BUILDINGS          3100000.0
35               39 TRANSPORTATION FACILITIES          3000000.0
34                       38 ASYLUMS AND HOMES          2875000.0
11                          11A CONDO-RENTALS          2750000.0
38  42 CONDO CULTURAL/MEDICAL/EDUCATIONAL/ETC          2640000.0
42                   46 C

In [33]:
property_type_value = (
    df[df["SALE PRICE"] > 0]
    .groupby("BUILDING CLASS CATEGORY").agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index().sort_values("MEDIAN_SALE_PRICE", ascending=False)
)

print(property_type_value.head(20))

                      BUILDING CLASS CATEGORY  TRANSACTION_COUNT  \
21                           25 LUXURY HOTELS                104   
30                                34 THEATRES                 53   
20                          23 LOFT BUILDINGS                 71   
7            08 RENTALS - ELEVATOR APARTMENTS               2768   
28          32 HOSPITAL AND HEALTH FACILITIES                238   
27                  31 COMMERCIAL VACANT LAND                127   
29                  33 EDUCATIONAL FACILITIES                288   
23                               27 FACTORIES               1125   
26                              30 WAREHOUSES               1788   
18                        21 OFFICE BUILDINGS               2295   
35               39 TRANSPORTATION FACILITIES                  9   
34                       38 ASYLUMS AND HOMES                126   
11                          11A CONDO-RENTALS                 30   
38  42 CONDO CULTURAL/MEDICAL/EDUCATIONAL/ETC   

In [34]:
property_type_value_filtered = (
    property_type_value[property_type_value["TRANSACTION_COUNT"] >= 500].sort_values("MEDIAN_SALE_PRICE", ascending=False)
)

print(property_type_value_filtered)

                       BUILDING CLASS CATEGORY  TRANSACTION_COUNT  \
7             08 RENTALS - ELEVATOR APARTMENTS               2768   
23                                27 FACTORIES               1125   
26                               30 WAREHOUSES               1788   
18                         21 OFFICE BUILDINGS               2295   
19                          22 STORE BUILDINGS               5343   
25                       29 COMMERCIAL GARAGES               1659   
33                     37 RELIGIOUS FACILITIES                631   
14                      14 RENTALS - 4-10 UNIT               3372   
6               07 RENTALS - WALKUP APARTMENTS              16238   
39                   43 CONDO OFFICE BUILDINGS                839   
10               11 SPECIAL CONDO BILLING LOTS               1201   
13             13 CONDOS - ELEVATOR APARTMENTS              80515   
2                    03 THREE FAMILY DWELLINGS              22152   
15           15 CONDOS - 2-10 UNIT

### Property Type Transaction Value

To compare transaction values across property types, median sale price was used instead of average sale price because the dataset contains extremely high-value transactions that can strongly influence the mean.

Property types were filtered to those with at least 500 positive-price transactions to reduce the influence of categories with very small sample sizes.

This threshold is used only for analysis and does not remove records from the cleaned dataset.

In [35]:
sales_by_year = (
    df.groupby("SALE YEAR").size().reset_index(name="TRANSACTION COUNT").sort_values("SALE YEAR")
)

print(sales_by_year)

   SALE YEAR  TRANSACTION COUNT
0       2016              80068
1       2017              79674
2       2018              76659
3       2019              61222
4       2020              51061
5       2021              69256
6       2022              65018
7       2023              53318
8       2024              54812
9       2025              59247


In [38]:
borough_value = (
    df[df["SALE PRICE"] > 0].groupby("BOROUGH")["SALE PRICE"].median()
    .reset_index(name="MEDIAN SALE PRICE").sort_values("MEDIAN SALE PRICE", ascending=False)
)

print(borough_value)

   BOROUGH  MEDIAN SALE PRICE
0        1          1470356.0
2        3           930000.0
3        4           695000.0
1        2           575000.0
4        5           565000.0


In [39]:
neighborhood_value = (
    df[df["SALE PRICE"] > 0].groupby("NEIGHBORHOOD").agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

neighborhood_value_filtered = (
    neighborhood_value[
        neighborhood_value["TRANSACTION_COUNT"] >= 500
    ]
    .sort_values("MEDIAN_SALE_PRICE", ascending=False)
)

print(neighborhood_value_filtered.head(20))

                  NEIGHBORHOOD  TRANSACTION_COUNT  MEDIAN_SALE_PRICE
130              JAVITS CENTER                545          4900000.0
46                CIVIC CENTER               1543          3230000.0
215                       SOHO               1976          2950000.0
109     GREENWICH VILLAGE-WEST               2804          2800000.0
233                    TRIBECA               3189          2672906.0
79                     FASHION                730          2400000.0
39             CARROLL GARDENS               1210          2373577.0
86                    FLATIRON               1881          2360000.0
51                 COBBLE HILL                778          2135000.0
154                MIDTOWN CBD               1316          1997000.0
31            BROOKLYN HEIGHTS               1852          1991250.0
22                 BOERUM HILL               1328          1930000.0
108  GREENWICH VILLAGE-CENTRAL               2156          1900000.0
42                     CHELSEA    

In [42]:
size_value = (
    df[
        (df["SALE PRICE"] > 0) &
        (df["GROSS SQUARE FEET"] > 0)
    ]
    .copy()
)

size_value["SIZE QUARTILE"] = pd.qcut(
    size_value["GROSS SQUARE FEET"],
    q=4,
    labels=["Smallest", "Lower-Mid", "Upper-Mid", "Largest"]
)

size_value_summary = (
    size_value
    .groupby("SIZE QUARTILE", observed=True)
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_GROSS_SQFT=("GROSS SQUARE FEET", "median"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

print(size_value_summary)

  SIZE QUARTILE  TRANSACTION_COUNT  MEDIAN_GROSS_SQFT  MEDIAN_SALE_PRICE
0      Smallest              10665              720.0           520000.0
1     Lower-Mid              10613              960.0           535000.0
2     Upper-Mid              10639             1512.0           615000.0
3       Largest              10618             2741.0           965000.0


In [44]:
unit_value = (
    df[(df["SALE PRICE"] > 0) & (df["TOTAL UNITS"] > 0)
       ].copy()
)

unit_value["UNIT GROUP"] = pd.cut(
    unit_value["TOTAL UNITS"],
    bins=[0, 1, 2, 5, 10, float("inf")],
    labels=["1 Unit", "2 Units", "3-5 Units", "6-10 Units", "11+ Units"]
)

unit_value_summary = (
    unit_value.groupby("UNIT GROUP", observed=True).agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_UNITS=("TOTAL UNITS", "median"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    ).reset_index()
)

print(unit_value_summary)

   UNIT GROUP  TRANSACTION_COUNT  MEDIAN_UNITS  MEDIAN_SALE_PRICE
0      1 Unit             237828           1.0           745000.0
1     2 Units              92111           2.0           800000.0
2   3-5 Units              36812           3.0          1090000.0
3  6-10 Units               8225           7.0          1825000.0
4   11+ Units               8633          29.0          4655000.0


In [45]:
age_value = (
    df[(df["SALE PRICE"] > 0) & (df["PROPERTY AGE"] >= 0)
       ].copy()
)

age_value["AGE GROUP"] = pd.cut(
    age_value["PROPERTY AGE"],
    bins=[-1, 10, 25, 50, 100, float("inf")],
    labels=[
        "0-10 Years",
        "11-25 Years",
        "26-50 Years",
        "51-100 Years",
        "100+ Years"
    ]
)

age_value_summary = (
    age_value.groupby("AGE GROUP", observed=True).agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_AGE=("PROPERTY AGE", "median"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    ).reset_index()
)

print(age_value_summary)

      AGE GROUP  TRANSACTION_COUNT  MEDIAN_AGE  MEDIAN_SALE_PRICE
0    0-10 Years              55006         4.0          1018249.0
1   11-25 Years              38120        16.0           790000.0
2   26-50 Years              42742        36.0           640000.0
3  51-100 Years             216576        83.0           700000.0
4    100+ Years              79217       111.0           969000.0


In [47]:
location_property_value = (
    df[
        (df["SALE PRICE"] > 0)
    ]
    .groupby(
        ["BOROUGH", "BUILDING CLASS CATEGORY"]
    )
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

location_property_value_filtered = (
    location_property_value[
        location_property_value["TRANSACTION_COUNT"] >= 500
    ]
    .sort_values(
        "MEDIAN_SALE_PRICE",
        ascending=False
    )
)

print(location_property_value_filtered.head(25))

     BOROUGH            BUILDING CLASS CATEGORY  TRANSACTION_COUNT  \
17         1                21 OFFICE BUILDINGS                798   
90         3   08 RENTALS - ELEVATOR APARTMENTS                609   
18         1                 22 STORE BUILDINGS                628   
0          1            01 ONE FAMILY DWELLINGS               1035   
13         1             14 RENTALS - 4-10 UNIT                516   
5          1     07 RENTALS - WALKUP APARTMENTS               3765   
6          1   08 RENTALS - ELEVATOR APARTMENTS               1412   
1          1            02 TWO FAMILY DWELLINGS                785   
155        4                      30 WAREHOUSES                599   
109        3                      30 WAREHOUSES                694   
14         1  15 CONDOS - 2-10 UNIT RESIDENTIAL               1670   
101        3                21 OFFICE BUILDINGS                543   
102        3                 22 STORE BUILDINGS               1794   
108        3        

In [48]:
location_property_value["VALUE_SCORE"] = (
    location_property_value["MEDIAN_SALE_PRICE"] * location_property_value["TRANSACTION_COUNT"]
)

location_property_opportunities = (
    location_property_value[location_property_value["TRANSACTION_COUNT"] >= 500]
    .sort_values("VALUE_SCORE", ascending=False)
)

print(
    location_property_opportunities[
        [
            "BOROUGH",
            "BUILDING CLASS CATEGORY",
            "TRANSACTION_COUNT",
            "MEDIAN_SALE_PRICE",
            "VALUE_SCORE"
        ]
    ].head(20)
)

     BOROUGH            BUILDING CLASS CATEGORY  TRANSACTION_COUNT  \
12         1    13 CONDOS - ELEVATOR APARTMENTS              45547   
129        4            01 ONE FAMILY DWELLINGS              55924   
84         3            02 TWO FAMILY DWELLINGS              32898   
130        4            02 TWO FAMILY DWELLINGS              32077   
173        5            01 ONE FAMILY DWELLINGS              33646   
96         3    13 CONDOS - ELEVATOR APARTMENTS              17840   
5          1     07 RENTALS - WALKUP APARTMENTS               3765   
83         3            01 ONE FAMILY DWELLINGS              21434   
17         1                21 OFFICE BUILDINGS                798   
8          1     10 COOPS - ELEVATOR APARTMENTS              16750   
85         3          03 THREE FAMILY DWELLINGS              10715   
89         3     07 RENTALS - WALKUP APARTMENTS               6987   
142        4    13 CONDOS - ELEVATOR APARTMENTS              15376   
174        5        

In [49]:
volume_threshold = location_property_value["TRANSACTION_COUNT"].median()
price_threshold = location_property_value["MEDIAN_SALE_PRICE"].median()

location_property_segments = (
    location_property_value[
        location_property_value["TRANSACTION_COUNT"] >= 500
    ]
    .copy()
)

location_property_segments["MARKET SEGMENT"] = np.select(
    [
        (location_property_segments["TRANSACTION_COUNT"] >= volume_threshold) &
        (location_property_segments["MEDIAN_SALE_PRICE"] >= price_threshold),

        (location_property_segments["TRANSACTION_COUNT"] >= volume_threshold) &
        (location_property_segments["MEDIAN_SALE_PRICE"] < price_threshold),

        (location_property_segments["TRANSACTION_COUNT"] < volume_threshold) &
        (location_property_segments["MEDIAN_SALE_PRICE"] >= price_threshold)
    ],
    [
        "High Volume / High Value",
        "High Volume / Lower Value",
        "Low Volume / High Value"
    ],
    default="Low Volume / Lower Value"
)

print("Transaction volume threshold:", volume_threshold)
print("Median sale price threshold:", price_threshold)

print("\nMarket segments:")
print(
    location_property_segments[
        [
            "BOROUGH",
            "BUILDING CLASS CATEGORY",
            "TRANSACTION_COUNT",
            "MEDIAN_SALE_PRICE",
            "MARKET SEGMENT"
        ]
    ]
    .sort_values(
        ["MARKET SEGMENT", "MEDIAN_SALE_PRICE"],
        ascending=[True, False]
    )
    .head(30)
)

Transaction volume threshold: 101.5
Median sale price threshold: 1466557.0

Market segments:
     BOROUGH            BUILDING CLASS CATEGORY  TRANSACTION_COUNT  \
17         1                21 OFFICE BUILDINGS                798   
90         3   08 RENTALS - ELEVATOR APARTMENTS                609   
18         1                 22 STORE BUILDINGS                628   
0          1            01 ONE FAMILY DWELLINGS               1035   
13         1             14 RENTALS - 4-10 UNIT                516   
5          1     07 RENTALS - WALKUP APARTMENTS               3765   
6          1   08 RENTALS - ELEVATOR APARTMENTS               1412   
1          1            02 TWO FAMILY DWELLINGS                785   
155        4                      30 WAREHOUSES                599   
109        3                      30 WAREHOUSES                694   
14         1  15 CONDOS - 2-10 UNIT RESIDENTIAL               1670   
101        3                21 OFFICE BUILDINGS                543 

In [51]:
zero_price_by_year = (
    df.groupby("SALE YEAR")
    .agg(
        TOTAL_TRANSACTIONS=("SALE PRICE", "count"),
        ZERO_PRICE_TRANSACTIONS=(
            "SALE PRICE",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

zero_price_by_year["ZERO_PRICE_PCT"] = (
    zero_price_by_year["ZERO_PRICE_TRANSACTIONS"] / zero_price_by_year["TOTAL_TRANSACTIONS"] * 100
)

print(zero_price_by_year)

   SALE YEAR  TOTAL_TRANSACTIONS  ZERO_PRICE_TRANSACTIONS  ZERO_PRICE_PCT
0       2016               80068                    23398       29.222661
1       2017               79674                    22852       28.681879
2       2018               76659                    22181       28.934633
3       2019               61222                    21151       34.548038
4       2020               51061                    20030       39.227591
5       2021               69256                    22203       32.059316
6       2022               65018                    21558       33.156972
7       2023               53318                    20379       38.221614
8       2024               54812                    21126       38.542655
9       2025               59247                    23796       40.164059
